# GA1-240201064-04-AA1-EV09
## Clasificacion de variables y seleccion de graficas (dataset del proyecto)

Dataset base: `dataset.csv`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
ARTIFACTS_DIR = (NOTEBOOK_DIR / '..' / 'artifacts' / 'ev09').resolve()
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    NOTEBOOK_DIR / '..' / 'data' / 'raw' / 'dataset.csv',
    NOTEBOOK_DIR / '..' / 'data' / 'Raw' / 'dataset.csv',
    NOTEBOOK_DIR / 'data' / 'raw' / 'dataset.csv',
    NOTEBOOK_DIR / 'data' / 'Raw' / 'dataset.csv',
]

DATASET_PATH = None
for p in candidate_paths:
    if p.exists():
        DATASET_PATH = p.resolve()
        break

if DATASET_PATH is None:
    raise FileNotFoundError('No se encontro dataset.csv en data/raw o data/Raw')

print('Dataset:', DATASET_PATH)
print('Artifacts:', ARTIFACTS_DIR)


Dataset: C:\Users\WILLY-A\Desktop\Ciencia de datos\proyecto_ds\data\Raw\dataset.csv
Artifacts: C:\Users\WILLY-A\Desktop\Ciencia de datos\proyecto_ds\artifacts\ev09


In [2]:
df = pd.read_csv(DATASET_PATH)
print('Shape:', df.shape)
df.head(3)


Shape: (16598, 11)


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82


## 1) Identificacion de variables del dataset

In [3]:
identificacion = pd.DataFrame({
    'Variable': df.columns,
    'Tipo_dato_pandas': [str(df[c].dtype) for c in df.columns],
    'Nulos': [int(df[c].isna().sum()) for c in df.columns],
    'Valores_unicos': [int(df[c].nunique(dropna=True)) for c in df.columns],
    'Ejemplo': [str(df[c].dropna().iloc[0]) if not df[c].dropna().empty else 'NA' for c in df.columns]
})

identificacion


,Variable,Tipo_dato_pandas,Nulos,Valores_unicos,Ejemplo
0,Rank,int64,0,16598,1
1,Name,object,0,11493,Wii Sports
2,Platform,object,0,31,Wii
3,Year,float64,271,39,2006.0
4,Genre,object,0,12,Sports
5,Publisher,object,58,578,Nintendo
6,NA_Sales,float64,0,409,41.49
7,EU_Sales,float64,0,305,29.02
8,JP_Sales,float64,0,244,3.77
9,Other_Sales,float64,0,157,8.46


## 2) Tabla de clasificacion (tipo, subtipo, justificacion)

In [4]:
clasificacion = {
    'Rank': ('Cuantitativa', 'Discreta', 'Representa una posicion numerica entera en el ranking.'),
    'Name': ('Cualitativa', 'Nominal', 'Nombre del videojuego; categorias sin orden.'),
    'Platform': ('Cualitativa', 'Nominal', 'Plataforma de juego; categorias sin jerarquia.'),
    'Year': ('Cuantitativa', 'Discreta', 'Ano de lanzamiento en valores enteros.'),
    'Genre': ('Cualitativa', 'Nominal', 'Genero del juego; categorias sin orden natural.'),
    'Publisher': ('Cualitativa', 'Nominal', 'Editorial del juego; categorias nominales.'),
    'NA_Sales': ('Cuantitativa', 'Continua', 'Ventas en millones con decimales posibles.'),
    'EU_Sales': ('Cuantitativa', 'Continua', 'Ventas en millones; variable medible continua.'),
    'JP_Sales': ('Cuantitativa', 'Continua', 'Ventas en millones; permite valores decimales.'),
    'Other_Sales': ('Cuantitativa', 'Continua', 'Ventas en otras regiones; magnitud continua.'),
    'Global_Sales': ('Cuantitativa', 'Continua', 'Suma de ventas globales en escala continua.'),
}

tabla_clasificacion = pd.DataFrame([
    {
        'Variable': col,
        'Tipo': clasificacion[col][0],
        'Subtipo': clasificacion[col][1],
        'Justificacion': clasificacion[col][2],
    }
    for col in df.columns
])

tabla_clasificacion


,Variable,Tipo,Subtipo,Justificacion
0,Rank,Cuantitativa,Discreta,Representa una posicion numerica entera en el ...
1,Name,Cualitativa,Nominal,Nombre del videojuego; categorias sin orden.
2,Platform,Cualitativa,Nominal,Plataforma de juego; categorias sin jerarquia.
3,Year,Cuantitativa,Discreta,Ano de lanzamiento en valores enteros.
4,Genre,Cualitativa,Nominal,Genero del juego; categorias sin orden natural.
5,Publisher,Cualitativa,Nominal,Editorial del juego; categorias nominales.
6,NA_Sales,Cuantitativa,Continua,Ventas en millones con decimales posibles.
7,EU_Sales,Cuantitativa,Continua,Ventas en millones; variable medible continua.
8,JP_Sales,Cuantitativa,Continua,Ventas en millones; permite valores decimales.
9,Other_Sales,Cuantitativa,Continua,Ventas en otras regiones; magnitud continua.


## 3) Tabla de seleccion de grafica recomendada por variable

In [5]:
def recomendar_grafica(tipo, subtipo, variable):
    if tipo == 'Cualitativa' and subtipo == 'Nominal':
        return 'Barras', 'Comparar frecuencias por categoria sin orden.'
    if tipo == 'Cuantitativa' and subtipo == 'Discreta':
        if variable == 'Year':
            return 'Linea', 'Permite ver la evolucion temporal de lanzamientos por ano.'
        return 'Barras', 'Conteo o comparacion por valores enteros.'
    if tipo == 'Cuantitativa' and subtipo == 'Continua':
        if variable in ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']:
            return 'Histograma y Boxplot', 'Permite analizar distribucion y valores atipicos de ventas.'
        return 'Histograma', 'Variable continua: conviene ver forma de distribucion.'
    return 'Barras', 'Visualizacion general recomendada.'

rows = []
for _, r in tabla_clasificacion.iterrows():
    graf, just = recomendar_grafica(r['Tipo'], r['Subtipo'], r['Variable'])
    rows.append({
        'Variable': r['Variable'],
        'Tipo': f"{r['Tipo']} {r['Subtipo']}",
        'Grafica_recomendada': graf,
        'Justificacion': just,
    })

tabla_graficas = pd.DataFrame(rows)
tabla_graficas


,Variable,Tipo,Grafica_recomendada,Justificacion
0,Rank,Cuantitativa Discreta,Barras,Conteo o comparacion por valores enteros.
1,Name,Cualitativa Nominal,Barras,Comparar frecuencias por categoria sin orden.
2,Platform,Cualitativa Nominal,Barras,Comparar frecuencias por categoria sin orden.
3,Year,Cuantitativa Discreta,Linea,Permite ver la evolucion temporal de lanzamien...
4,Genre,Cualitativa Nominal,Barras,Comparar frecuencias por categoria sin orden.
5,Publisher,Cualitativa Nominal,Barras,Comparar frecuencias por categoria sin orden.
6,NA_Sales,Cuantitativa Continua,Histograma y Boxplot,Permite analizar distribucion y valores atipic...
7,EU_Sales,Cuantitativa Continua,Histograma y Boxplot,Permite analizar distribucion y valores atipic...
8,JP_Sales,Cuantitativa Continua,Histograma y Boxplot,Permite analizar distribucion y valores atipic...
9,Other_Sales,Cuantitativa Continua,Histograma y Boxplot,Permite analizar distribucion y valores atipic...


In [6]:
# Guardar tablas como soporte de evidencia
tabla_clasificacion.to_csv(ARTIFACTS_DIR / 'tabla_clasificacion_ev09.csv', index=False, encoding='utf-8-sig')
tabla_graficas.to_csv(ARTIFACTS_DIR / 'tabla_graficas_ev09.csv', index=False, encoding='utf-8-sig')
print('Tablas exportadas en:', ARTIFACTS_DIR)


Tablas exportadas en: C:\Users\WILLY-A\Desktop\Ciencia de datos\proyecto_ds\artifacts\ev09


## 4) Reflexion final

1. **Que variable fue mas dificil de clasificar y por que?**
`Rank` fue la mas discutible, porque es numerica pero su uso es de posicion. Se clasifico como cuantitativa discreta por su naturaleza entera.

2. **Que riesgo hay al escoger una grafica incorrecta?**
Se pueden interpretar mal patrones y tomar decisiones equivocadas, por ejemplo comparar proporciones con un grafico no adecuado.

3. **Como impacta la clasificacion de variables en la analitica?**
Define que metricas, pruebas y visualizaciones tienen sentido; una mala clasificacion afecta toda la lectura del analisis.
